# Suite INT — Interactions

`Interactions` (`mbse.Architecture.Systems.Framework.Interactions`) defines the library of modules, their interfaces
and ports, implementations of a module by instances and channels, the schema of a channel's messages, and how a
channel is attached. These cases cover the schemas `define` builds, the meta-schemas it extends, the data built with
them, and the functions that read it: `path`, `module_of`, `message_schema`, `distribution` and `ends`.

In [ ]:
import dataclasses

from mbse.Schemas.Framework import Builders, Validators
from mbse.Architecture.Systems.Examples import Pipeline
from mbse.Architecture.Systems.Framework import Interactions, Structure
from support import raises, same

## INT-01 · `define` builds the Interactions schemas once, after Structure's, and extends the meta-schemas

In [ ]:
store = Builders.Store()
I = Interactions.define(store)
names = [f.name for f in dataclasses.fields(I)]
assert names == ["Library", "Module", "Interface", "Port", "Implementation", "Instance", "Channel", "HasModule",
                 "HasInterface", "HasPort", "Implements", "Instantiates", "HasInstance", "HasChannel", "HasSchema",
                 "Connects"]
assert [getattr(I, n).name for n in names] == [f"Interactions.{n}" for n in names]
assert all(getattr(I, n).validate() == [] and getattr(I, n).description for n in names)
again = Interactions.define(store)  # defined again, the store gives the schemas it holds
assert all(getattr(again, n) is getattr(I, n) for n in names)

# Structure's schemas are defined with them; implementations and instances are containers.
S = Structure.define(store)
assert all(schema.refines is S.Container for schema in (I.Implementation, I.Instance))
assert list(I.Implementation.properties) == ["name", "description"]
assert list(I.Module.properties) == ["description"]  # a module's name is its HasModule entry's
assert list(I.Port.properties) == ["direction", "description"]
assert list(I.Channel.properties) == ["distribution", "description"]
assert I.Library.singleton == "Interactions.Library" and store.singleton("Interactions.Library").as_schema() is I.Library
owned = {(s.name, a): adjacency.owner for s in (I.Module, I.Interface, I.Port, I.Instance, I.Channel)
         for a, adjacency in s.adjacencies.items()}
assert [k for k, owner in owned.items() if owner] == [
    ("Interactions.Module", "library"), ("Interactions.Interface", "module"), ("Interactions.Port", "interface")]

# Each meta-schema of a value lets its schemas fill HasSchema's schema link, once, however often define runs.
for kind in Interactions.MESSAGE_KINDS:
    meta = getattr(store.Schemas, kind).as_schema()
    assert [a.me for a in meta.anonymous if a.relation is I.HasSchema] == ["schema"], kind
assert not any(a.relation is I.HasSchema for a in store.Schemas.OfRelation.as_schema().anonymous)

# A store that holds some of the names, but not all, is refused.
other = Builders.Store()
with other.transaction():
    other.Schemas.OfObject(lambda o: o.name("Interactions.Module").ref(True))
with raises(ValueError, match="holds some of the Interactions schemas"):
    Interactions.define(other)

## INT-02 · Modules in the library, their interfaces and ports by name, each with a path; none outside it

In [ ]:
store = Builders.Store()
I = Interactions.define(store)
library = store.singleton("Interactions.Library")
with store.transaction():
    store.Interactions.Library(library, lambda lib: lib.modules(lambda a: a.insert(lambda e: e.name("Uart").module(
        lambda m: m.description("A serial port").interfaces(lambda a: a.insert(lambda e: e.name("serial").intf(
            lambda i: i.ports(lambda p: (
                p
                .insert(lambda e: e.name("tx").port(lambda q: q.direction("out")))
                .insert(lambda e: e.name("rx").port(lambda q: q.direction("in")))
            ))
        )))
    ))))
uart = library.modules.at("Uart").module
serial = uart.interfaces.at("serial").intf
tx, rx = serial.ports.at("tx").port, serial.ports.at("rx").port
assert [str(e.name) for e in serial.ports] == ["tx", "rx"] and [str(p.direction) for p in (tx, rx)] == ["out", "in"]
assert str(uart.description) == "A serial port"
assert same([e.module for e in serial.module], [uart]) and same([e.intf for e in tx.interface], [serial])

# Each has a path from the library, by which the store finds it, and a dotted name.
assert store.Instances.Interactions.Library.modules({"name": "Uart"}).interfaces({"name": "serial"}).ports(
    {"name": "tx"}) is tx
assert [Interactions.path(x) for x in (uart, serial, tx)] == ["Uart", "Uart.serial", "Uart.serial.tx"]
assert Interactions.path(library) is None

# The library owns every module: one outside it is refused when its transaction ends, with all it wrote.
with raises(ValueError, match="needs an owner: the key 'name' of 'Interactions.HasModule'"):
    with store.transaction():
        store.Interactions.Module(lambda m: m.description("Not in the library"))
assert [str(e.name) for e in library.modules] == ["Uart"]

# A name finds one interface: a second under the same name is HasInterface's unique clause's problem.
with store.transaction():
    store.Interactions.Module(uart, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("serial").intf(lambda i: i))))
assert any(p.startswith("unique(intf) violated") for p in Validators.Validate(store).Reachable(I.Module, uart))

## INT-03 · Implementations, their instances, and the module each is of

In [ ]:
store = Builders.Store()
p = Pipeline.build(store)
I = p.schemas
assert Interactions.module_of(p.impl) is p.pipeline
assert same([Interactions.module_of(p.instance[n]) for n in ("sensor", "filter", "logger")], [p.sensor, p.filter, p.logger])
assert [str(e.name) for e in p.impl.instances] == ["sensor", "filter", "logger"]
assert [str(e.name) for e in p.impl.channels] == ["raw", "filtered"]
assert same([e.instance for e in p.sensor.instances], [p.instance["sensor"]])
assert same([e.impl for e in p.pipeline.implementations], [p.impl])
assert [Interactions.path(x) for x in (p.impl, p.instance["filter"], p.channel["raw"])] == [
    "reference", "reference.filter", "reference.raw"]

# An implementation without a name, and what it holds, have no path; a module of none is None.
with store.transaction():
    bare = store.Interactions.Implementation(lambda x: x.instances(lambda a: a.insert(lambda e: e.name("only").instance(
        lambda i: i))))
assert Interactions.path(bare) is None and Interactions.path(bare.instances.at("only").instance) is None
assert Interactions.module_of(bare) is None

# An instance is of one module: a second is Instantiates' unique clause's problem.
with store.transaction():
    store.Interactions.Instance(p.instance["logger"], lambda i: i.module(lambda a: a.insert(lambda e: e.module(p.sensor))))
assert any(x.startswith("unique(module) violated") for x in Validators.Validate(store).Reachable(I.Module, p.pipeline))

## INT-04 · A channel carries messages of a schema of any kind, linked as the schema object itself

In [ ]:
store = Builders.Store()
I = Interactions.define(store)
Int = store.Schemas.OfInt
with store.transaction():
    Packet = store.Schemas.OfObject(lambda o: o.name("bus.Packet").properties([lambda t: t.name("length").schema([Int])]))
    Word = store.Schemas.OfContainer(lambda c: c.name("bus.Word").index([Int]).element([store.Schemas.OfBool]))
    Command = store.Schemas.OfUnion(lambda u: u.name("bus.Command").branches([
        lambda t: t.name("read").schema([Int]),
        lambda t: t.name("write").schema([Packet]),
    ]))
    Tagged = store.Schemas.OfIntersection(lambda i: i.name("bus.Tagged").parts([lambda t: t.name("packet").schema([Packet])]))
    Anything = store.Schemas.OfAny(lambda a: a.name("bus.Anything"))
    Link = store.Schemas.OfRelation(lambda r: r.name("bus.Link").links([lambda t: t.name("a"), lambda t: t.name("b")]))

for schema in (Int.as_schema(), Packet, Word, Command, Tagged, Anything):
    with store.transaction():
        channel = store.Interactions.Channel(lambda c: c.message(lambda a: a.insert(lambda e: e.schema(schema))))
    assert Interactions.message_schema(channel) is schema
    assert Validators.Validate(store)(I.Channel, channel) == [], schema.name

# A relation describes no value: it cannot be a channel's message schema.
with store.transaction():
    channel = store.Interactions.Channel(lambda c: c.message(lambda a: a.insert(lambda e: e.schema(Link))))
assert Validators.Validate(store)(I.Channel, channel) == [
    "Interactions.Channel#0.message[0].schema: a 'Schemas.OfRelation' cannot fill link 'schema'; its schema declares "
    "no adjacency, named or anonymous, to this relation via 'schema'",
]

# A channel without one has none; its distribution is BROADCAST unless it says otherwise.
with store.transaction():
    blank = store.Interactions.Channel(lambda c: c)
    anycast = store.Interactions.Channel(lambda c: c.distribution("ANYCAST"))
assert Interactions.message_schema(blank) is None
assert Interactions.distribution(blank) == Interactions.BROADCAST == "BROADCAST" and blank.distribution.get() is None
assert Interactions.distribution(anycast) == "ANYCAST"
assert Interactions.DISTRIBUTIONS == ("BROADCAST", "ANYCAST", "ROUND_ROBIN")

## INT-05 · A channel's ends: an instance's out port is a source, and at the boundary the other way round

In [ ]:
store = Builders.Store()
p = Pipeline.build(store)
assert [(Interactions.path(e.instance), Interactions.path(e.intf), Interactions.path(e.port), e.role)
        for e in Interactions.ends(p.channel["filtered"])] == [
    ("reference.filter", "Filter.filtered", "Filter.filtered.tx", "source"),
    ("reference.logger", "Logger.records", "Logger.records.rx", "destination"),
    ("reference", "Pipeline.readings", "Pipeline.readings.out", "destination"),
]

# The pipeline's own in port, at the boundary, is a source; a port without a direction gives no end.
with store.transaction():
    store.Interactions.Module(p.pipeline, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("settings").intf(
        lambda i: i.ports(lambda a: a.insert(lambda e: e.name("in").port(lambda q: q.direction("in"))))))))
    store.Interactions.Module(p.filter, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("settings").intf(
        lambda i: i.ports(lambda a: a.insert(lambda e: e.name("rx").port(lambda q: q)))))))
    store.Interactions.Implementation(p.impl, lambda x: x.channels(lambda a: a.insert(lambda e: e.name("settings").channel(
        lambda c: c))))
    settings = p.impl.channels.at("settings").channel
    Pipeline.connect(store, settings, p.impl, "settings", "in")
    Pipeline.connect(store, settings, p.instance["filter"], "settings", "rx")
assert [(Interactions.path(e.port), e.role) for e in Interactions.ends(settings)] == [
    ("Pipeline.settings.in", "source"), ("Filter.settings.rx", None)]
assert all(isinstance(e, Interactions.End) for e in Interactions.ends(settings))